In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "historic_tmax_market_data"
CSV_PATTERN = "*/*tmax_kalshi*5min_same_day.csv"
FROZEN_K = 4  # From: python src/snapshot_stability.py

csv_paths = sorted(RAW_DATA_DIR.glob(CSV_PATTERN))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under {RAW_DATA_DIR}")

frames = []
for path in csv_paths:
    city_df = pd.read_csv(path)
    if "city" not in city_df.columns:
        city_df["city"] = path.parent.name.replace("_", " ").title()
    city_df["event_date"] = pd.to_datetime(city_df["event_date"]).dt.date.astype(str)
    city_df["snapshot_time_local"] = pd.to_datetime(city_df["snapshot_time_local"])
    city_df["source_city_folder"] = path.parent.name
    frames.append(city_df)

market_df = pd.concat(frames, ignore_index=True)
market_df = market_df.sort_values(["city", "event_date", "snapshot_time_local", "bucket_label"])
print(f"Loaded {len(market_df):,} rows from {len(csv_paths)} city CSVs")
market_df.head()

Loaded 1,537,620 rows from 12 city CSVs


,event_date,station,nws_tmax_f,nws_tmax_time_local,nws_tmax_time_local_hhmm,nws_report_issue_utc,nws_report_file,nws_report_excerpt_maximum,series_ticker,event_ticker,...,no_mid_close,last_trade_price_close,last_trade_price_previous,volume_contracts,open_interest_contracts,city,source_city_folder,station_text,nws_pil,nws_product_id
1,2026-03-30,KAUS,89,2026-03-30T16:08:00-05:00,16:08,2026-03-31T06:55:00+00:00,/Users/oscaro/Desktop/Commodities_Trading/Stat...,MAXIMUM 89 4:08 PM,KXHIGHAUS,KXHIGHAUS-26MAR30,...,0.790,0.3,0.15,1.0,29.0,Austin,austin,NaN,NaN,NaN
2,2026-03-30,KAUS,89,2026-03-30T16:08:00-05:00,16:08,2026-03-31T06:55:00+00:00,/Users/oscaro/Desktop/Commodities_Trading/Stat...,MAXIMUM 89 4:08 PM,KXHIGHAUS,KXHIGHAUS-26MAR30,...,0.620,NaN,0.15,0.0,28.0,Austin,austin,NaN,NaN,NaN
3,2026-03-30,KAUS,89,2026-03-30T16:08:00-05:00,16:08,2026-03-31T06:55:00+00:00,/Users/oscaro/Desktop/Commodities_Trading/Stat...,MAXIMUM 89 4:08 PM,KXHIGHAUS,KXHIGHAUS-26MAR30,...,0.905,NaN,NaN,0.0,0.0,Austin,austin,NaN,NaN,NaN
4,2026-03-30,KAUS,89,2026-03-30T16:08:00-05:00,16:08,2026-03-31T06:55:00+00:00,/Users/oscaro/Desktop/Commodities_Trading/Stat...,MAXIMUM 89 4:08 PM,KXHIGHAUS,KXHIGHAUS-26MAR30,...,0.940,NaN,0.03,0.0,8.0,Austin,austin,NaN,NaN,NaN
0,2026-03-30,KAUS,89,2026-03-30T16:08:00-05:00,16:08,2026-03-31T06:55:00+00:00,/Users/oscaro/Desktop/Commodities_Trading/Stat...,MAXIMUM 89 4:08 PM,KXHIGHAUS,KXHIGHAUS-26MAR30,...,0.855,NaN,0.15,0.0,5.0,Austin,austin,NaN,NaN,NaN


In [2]:
from src.market_explorer_widgets import display_market_explorer

market_explorer = display_market_explorer(market_df, frozen_k=FROZEN_K)